# Single-cell and single-nucleus molecular phenotype preprocessing

This mini-protocol performs cell-level snRNA-seq quality control and annotation, then prepares cell-type pseudobulk molecular phenotypes for QTL analysis.

#### Miniprotocol Timing

This is the total duration for one selected route; module-specific timings appear on their respective pages.

Timing: TBD

## Overview

Single-cell and single-nucleus data require two distinct levels of processing. Cell-level processing filters low-quality cells, removes ambient RNA and doublets, and transfers cell-type labels. Pseudobulk processing then aggregates cells within each donor and cell type, harmonizes donor identifiers, adjusts technical covariates, and formats residualized traits for QTL analysis.

The first two steps use [`snRNAseq_preprocessing.ipynb`](snRNAseq_preprocessing.ipynb). The remaining steps use [`pseudobulk_preprocessing.ipynb`](pseudobulk_preprocessing.ipynb). Choose a route from the table rather than running every command automatically.

## Steps

| Analysis goal | Commands to run, in order | Inputs |
|---|---:|---|
| Cell-level QC and reference-based annotation | 1 → 2 | `input/snrnaseq/protocol_example.snrnaseq.cellranger/`<br>`input/snrnaseq/protocol_example.snrnaseq.id_mapping.csv`<br>`input/snrnaseq/protocol_example.snrnaseq.seurat_ref_SE.rds` |
| Generate pseudobulk counts for a selected cell type from the annotated object | 1 → 2 → 3 | Inputs above |
| Prepare existing RNA pseudobulk counts for QTL analysis | 4 → 5 → 6 | `input/snrnaseq/protocol_example.snrnaseq.id_map.csv`<br>`input/snrnaseq/protocol_example.snrnaseq.metadata_MIC.csv`<br>`input/snrnaseq/protocol_example.snrnaseq.pseudobulk_counts_MIC.csv.gz`<br>`input/snrnaseq/protocol_example.snrnaseq.tech_vars_MIC.csv`<br>`input/reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.ERCC.gtf` |
| Format existing cell-type residuals for QTL analysis | 6 only | A residual matrix under a cell-type directory and the gene-annotation GTF |

The commands use `MIC` as the concrete example cell type. For another cell type, replace `MIC` consistently in the option value and associated input and output filenames.

Step 3 produces pseudobulk counts but not the metadata and technical-variable files required by steps 4 and 5. Continue with 4 → 5 → 6 only after those companion files are available.

### 1. [Cell-level quality control](snRNAseq_preprocessing.ipynb)

**What it does:** Filters cells, removes ambient RNA and doublets, and saves the filtered Seurat object plus a QC table.

In [ ]:
sos run pipeline/snRNAseq_preprocessing.ipynb sctk_qc  --input-dir input/snrnaseq/protocol_example.snrnaseq.cellranger  --output-dir output/snrna_seq  --sample-meta input/snrnaseq/protocol_example.snrnaseq.id_mapping.csv  --name protocol_example.snrnaseq

### 2. [Cell-type annotation](snRNAseq_preprocessing.ipynb)

**What it does:** Transfers labels from the reference Seurat object to the filtered cells and writes a cell-typed Seurat object.

In [ ]:
sos run pipeline/snRNAseq_preprocessing.ipynb cell_annotation  --sctk-rds output/snrna_seq/SCTK_results/protocol_example.snrnaseq.filtered_seuratobj.rds  --output-dir output/snrna_seq  --seurat-ref input/snrnaseq/protocol_example.snrnaseq.seurat_ref_SE.rds  --name protocol_example.snrnaseq

### 3. [Pseudobulk count aggregation](pseudobulk_preprocessing.ipynb)

**What it does:** Aggregates counts across cells of the selected type within each donor from the annotated Seurat object.

In [ ]:
sos run pipeline/pseudobulk_preprocessing.ipynb pseudobulk_counts  --seurat-files output/snrna_seq/protocol_example.snrnaseq.celltyped_seuratobj.rds  --celltype MIC  --output-dir output/snrna_seq  --name protocol_example

### 4. [Sample-ID harmonization](pseudobulk_preprocessing.ipynb)

**What it does:** Applies the donor-ID map to existing cell-type metadata and count matrices so their sample identifiers agree.

In [ ]:
sos run pipeline/pseudobulk_preprocessing.ipynb sampleid_mapping  --map-file input/snrnaseq/protocol_example.snrnaseq.id_map.csv  --meta-files input/snrnaseq/protocol_example.snrnaseq.metadata_MIC.csv  --count-files input/snrnaseq/protocol_example.snrnaseq.pseudobulk_counts_MIC.csv.gz  --output-dir output/snrna_seq  --name protocol_example

### 5. [Pseudobulk quality control and residualization](pseudobulk_preprocessing.ipynb)

**What it does:** Filters low-information features and samples, adjusts the specified technical variables, and writes cell-type residuals.

In [ ]:
sos run pipeline/pseudobulk_preprocessing.ipynb pseudobulk_qc  --meta-files output/snrna_seq/protocol_example.snrnaseq.metadata_MIC.csv  --count-files output/snrna_seq/protocol_example.snrnaseq.pseudobulk_counts_MIC.csv.gz  --tech-vars-file input/snrnaseq/protocol_example.snrnaseq.tech_vars_MIC.csv  --output-dir output/snrna_seq  --name protocol_example

### 6. [QTL phenotype formatting](pseudobulk_preprocessing.ipynb)

**What it does:** Adds genomic coordinates to the residualized cell-type traits and writes a bgzipped phenotype BED for association testing.

In [ ]:
sos run pipeline/pseudobulk_preprocessing.ipynb phenotype_formatting  --residual-files output/snrna_seq/protocol_example.MIC.residuals.txt  --output-dir output/snrna_seq  --gtf-file input/reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.ERCC.gtf  --name protocol_example

## Output Files

| Step | Relative path | Contents |
|---:|---|---|
| 1 | `output/snrna_seq/SCTK_results/protocol_example.snrnaseq.filtered_seuratobj.rds` | Filtered and corrected Seurat object |
| 1 | `output/snrna_seq/QC_table/protocol_example.snrnaseq.SCTK_QC_table.csv` | Per-cell QC measurements and calls |
| 2 | `output/snrna_seq/protocol_example.snrnaseq.celltyped_seuratobj.rds` | Seurat object with transferred cell-type labels |
| 3 | `output/snrna_seq/protocol_example.pseudobulk_counts_MIC.csv.gz` | Donor-by-feature cell-type pseudobulk counts |
| 4 | `output/snrna_seq/protocol_example.snrnaseq.metadata_MIC.csv` | Metadata with harmonized donor IDs |
| 4 | `output/snrna_seq/protocol_example.snrnaseq.pseudobulk_counts_MIC.csv.gz` | Counts with harmonized donor IDs |
| 5 | `output/snrna_seq/protocol_example.MIC.residuals.txt` | QC-filtered and residualized cell-type traits |
| 6 | `output/snrna_seq/protocol_example.MIC.phenotype.bed.gz` | Genomically positioned cell-type molecular phenotypes for QTL testing |

## Anticipated Results

The cell-level route produces a filtered, cell-typed Seurat object suitable for downstream aggregation. The pseudobulk route produces one donor-level molecular phenotype matrix per cell type and, after residualization and coordinate formatting, a bgzipped BED suitable for the phenotype-preprocessing and QTL-association workflows.

Review cell counts, donor representation, QC exclusions, and residual distributions before association testing; pseudobulk estimates can be unstable when a donor contributes too few cells to a cell type.

## Command interface

Inspect all workflows, options, and defaults for the two modules:

In [ ]:
sos run pipeline/snRNAseq_preprocessing.ipynb -h

In [ ]:
sos run pipeline/pseudobulk_preprocessing.ipynb -h